In [1]:
import scanpy as sc
import numpy as np

In [2]:
import re

def make_valid_names(names_vector):
    """
    Convert drug names to valid Python variable names by replacing invalid characters
    and ensuring the name starts with a letter or underscore.
    """
    valid_names = []
    for name in names_vector:
        # Replace invalid characters with underscores
        cleaned = re.sub(r'[^0-9a-zA-Z_]', '_', name)
        
        # Ensure the name starts with a letter or underscore
        if not re.match(r'^[a-zA-Z_]', cleaned):
            cleaned = f'X_{cleaned}'
        
        valid_names.append(cleaned)
    return valid_names

def clean_column_names(names_vector):
    """
    Remove 'factor(valid_drug_names)X' prefix and replace multiple underscores with a single underscore.
    """
    cleaned_names = []
    for name in names_vector:
        # Remove specific prefix
        name = re.sub(r'^factor\(valid_drug_names\)X', '', name)
        
        # Replace multiple underscores with a single underscore
        name = re.sub(r'_+', '_', name)
        
        cleaned_names.append(name)
    return cleaned_names

def clean_contrast_names(names_vector):
    """
    Clean contrast names by removing leading underscores and ensuring valid variable names.
    """
    cleaned_names = []
    for name in names_vector:
        # Remove leading underscores
        name = re.sub(r'^_+', '', name)
        
        # Ensure it starts with a letter or underscore
        if not re.match(r'^[a-zA-Z_]', name):
            name = f'X_{name}'
        
        # Replace non-alphanumeric characters with underscores
        name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
        
        cleaned_names.append(name)
    return cleaned_names

In [3]:
# Sciplex

In [4]:
file0 = '/lustre/groups/ml01/workspace/manuel.gander/sciplex/'

In [5]:
adata = sc.read_h5ad(file0+'bulked.h5ad')

/home/icb/manuel.gander/miniconda3/envs/seacells/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [6]:
conditions = sorted(set(adata.obs['drugname_drugconc']))

In [7]:
valid_names = make_valid_names(conditions)
column_cleaned = clean_column_names(valid_names)
contrast_cleaned = clean_contrast_names(column_cleaned)

In [8]:
D_exp = dict(zip(conditions, contrast_cleaned))

In [9]:
D_exp_inv = dict(zip(['factor_valid_drug_names_'+a for a in contrast_cleaned], conditions))

In [31]:
D_exp_inv["JQ1_10_0"] = "(+)-JQ1_10.0"
D_exp_inv["JQ1_100_0"] = "(+)-JQ1_100.0"
D_exp_inv["JQ1_1000_0"] = "(+)-JQ1_1000.0"
D_exp_inv["JQ1_10000_0"] = "(+)-JQ1_10000.0"

D_exp_inv["X2_Methoxyestradiol_2_MeOE2_10_0"] = "2-Methoxyestradiol (2-MeOE2)_10.0"
D_exp_inv["X2_Methoxyestradiol_2_MeOE2_100_0"] = "2-Methoxyestradiol (2-MeOE2)_100.0"
D_exp_inv["X2_Methoxyestradiol_2_MeOE2_1000_0"] = "2-Methoxyestradiol (2-MeOE2)_1000.0"
D_exp_inv["X2_Methoxyestradiol_2_MeOE2_10000_0"] = "2-Methoxyestradiol (2-MeOE2)_10000.0"

D_exp_inv["AV_939_10_0"] = "XAV-939_10.0"
D_exp_inv["AV_939_100_0"] = "XAV-939_100.0"
D_exp_inv["AV_939_1000_0"] = "XAV-939_1000.0"
D_exp_inv["AV_939_10000_0"] = "XAV-939_10000.0"

In [10]:
import pandas as pd
cellline = "A549"
output_file = f"/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/sciplex2/{cellline}_differential_expression_results.parquet"
df0 = pd.read_parquet(output_file)

In [11]:
df0['perturbation'] = df0.condition.map(D_exp_inv)

In [12]:
df0['perturbation'] = [a if 'plate' in a else b for a,b in zip(df0.condition, df0.perturbation)]

In [13]:
np.sum([str(a)=='nan' for a in df0.perturbation])

0

In [14]:
df = pd.DataFrame(D_exp_inv, index=range(1)).T

In [15]:
df.to_csv('Experiment_dict_Sciplex2.csv')